<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Chapter 2: Working with Text Data

Packages that are being used in this notebook:

In [203]:
from importlib.metadata import version

print("torch version", version("torch"))
print("tiktoken version", version("tiktoken"))

# Expected output:
# torch version: 2.5.1
# tiktoken version: 0.7.0

torch version 2.5.1
tiktoken version 0.13.0


- This chapter covers data preparation and sampling to get input data "ready" for the LLM

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="500px">

&nbsp;
## 2.1 Understanding word embeddings

- No code in this section

- There are many forms of embeddings; we focus on text embeddings in this book

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="500px">

- LLMs work with embeddings in high-dimensional spaces (i.e., thousands of dimensions)
- Since we can't visualize such high-dimensional spaces (we humans think in 1, 2, or 3 dimensions), the figure below illustrates a 2-dimensional embedding space

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="300px">

&nbsp;
## 2.2 Tokenizing text

- In this section, we tokenize text, which means breaking text into smaller units, such as individual words and punctuation characters

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="300px">

- Load raw text we want to work with
- [The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict) is a public domain short story

In [204]:
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)


<br>

---

<br>

#### Troubleshooting SSL certificate errors

- Some readers reported seeing ssl.SSLCertVerificationError: `SSL: CERTIFICATE_VERIFY_FAILED` when running `urllib.request.urlretrieve` in VSCode or Jupyter. 
- This usually means Python's certificate bundle is outdated.


**Fixes**

- Use Python ≥ 3.9; you can check your Python version by executing the following code:
```python
import sys
print(sys.__version__)
```
- Upgrade the cert bundle:
  - pip: `pip install --upgrade certifi`
  - uv: `uv pip install --upgrade certifi`
- Restart the Jupyter kernel after upgrading.
- If you still encounter an `ssl.SSLCertVerificationError` when executing the previous code cell, please see the discussion at [more information here on GitHub](https://github.com/rasbt/LLMs-from-scratch/pull/403)

<br>

---

<br>

In [205]:
with open("the-verdict.txt") as f:
    text = f.read()

print("Total number of character: ", len(text))
print(text[:99])

# Expected output:
# Total number of character: 20479
# I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 

Total number of character:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- The goal is to tokenize and embed this text for an LLM
- Let's develop a simple tokeniser based on some simple sample text that we can then later apply to the text above
- The following regular expression will split on whitespaces

In [206]:
import re

sample = "Hello world. This is a test."
re.split(r"(\s)", sample)

# Expected output:
# ['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']

['Hello', ' ', 'world.', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']

- We don't only want to split on whitespaces but also commas and periods, so let's modify the regular expression to do that as well

In [207]:
re.split(r'(\s|[,.])', sample)
# Expected output:
# ['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']

['Hello',
 ' ',
 'world',
 '.',
 '',
 ' ',
 'This',
 ' ',
 'is',
 ' ',
 'a',
 ' ',
 'test',
 '.',
 '']

- As we can see, this creates empty strings, let's remove them

In [208]:
# Strip whitespace from each item and then filter out any empty strings.

[s for s in re.split(r'(\s|[,.])', sample) if s.strip()]

# Expected output:
# ['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']

['Hello', 'world', '.', 'This', 'is', 'a', 'test', '.']

- This looks pretty good, but let's also handle other types of punctuation, such as periods, question marks, and so on

In [209]:
sample = "Hello, world. Is this -- a test?"
[s for s in re.split(r'(\s|[,.?]|--)', sample) if s.strip()]


# Expected output:
# ['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']

- This is pretty good, and we are now ready to apply this tokenization to the raw text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="350px">

In [210]:
preprocessed = [s for s in re.split(r'(\s|[,.:;?_!"()\']|--)', text) if s.strip()]
print(preprocessed)

# Expected output:
# ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in', 'the', 'height', 'of', 'his', 'glory', ',', 'he', 'had', 'dropped', 'his', 'painting', ',', 'married', 'a', 'rich', 'widow', ',', 'and', 'established', 'himself', 'in', 'a', 'villa', 'on', 'the', 'Riviera', '.', '(', 'Though', 'I', 'rather', 'thought', 'it', 'would', 'have', 'been', 'Rome', 'or', 'Florence', '.', ')', '"', 'The', 'height', 'of', 'his', 'glory', '"', '--', 'that', 'was', 'what', 'the', 'women', 'called', 'it', '.', 'I', 'can', 'hear', 'Mrs', '.', 'Gideon', 'Thwing', '--', 'his', 'last', 'Chicago', 'sitter', '--', 'deploring', 'his', 'unaccountable', 'abdication', '.', '"', 'Of', 'course', 'it', "'", 's', 'going', 'to', 'send', 'the', 'value', 'of', 'my', 'picture', "'", 'way', 'up', ';', 'but', 'I', 'don', "'", 't', 'think', 'of', 'that', ',

- Let's calculate the total number of tokens

In [211]:
len(preprocessed)

# Expected output:
# 4690

4690

&nbsp;
## 2.3 Converting tokens into token IDs

- Next, we convert the text tokens into token IDs that we can process via embedding layers later

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

- From these tokens, we can now build a vocabulary that consists of all the unique tokens

In [220]:
all_tokens = sorted(set(preprocessed))
len(all_tokens)
# print(enumerate(sorted(tokens)))

# Expected output:
# 1130

1130

In [221]:
all_tokens

['!',
 '"',
 "'",
 '(',
 ')',
 ',',
 '--',
 '.',
 ':',
 ';',
 '?',
 'A',
 'Ah',
 'Among',
 'And',
 'Are',
 'Arrt',
 'As',
 'At',
 'Be',
 'Begin',
 'Burlington',
 'But',
 'By',
 'Carlo',
 'Chicago',
 'Claude',
 'Come',
 'Croft',
 'Destroyed',
 'Devonshire',
 'Don',
 'Dubarry',
 'Emperors',
 'Florence',
 'For',
 'Gallery',
 'Gideon',
 'Gisburn',
 'Gisburns',
 'Grafton',
 'Greek',
 'Grindle',
 'Grindles',
 'HAD',
 'Had',
 'Hang',
 'Has',
 'He',
 'Her',
 'Hermia',
 'His',
 'How',
 'I',
 'If',
 'In',
 'It',
 'Jack',
 'Jove',
 'Just',
 'Lord',
 'Made',
 'Miss',
 'Money',
 'Monte',
 'Moon-dancers',
 'Mr',
 'Mrs',
 'My',
 'Never',
 'No',
 'Now',
 'Nutley',
 'Of',
 'Oh',
 'On',
 'Once',
 'Only',
 'Or',
 'Perhaps',
 'Poor',
 'Professional',
 'Renaissance',
 'Rickham',
 'Riviera',
 'Rome',
 'Russian',
 'Sevres',
 'She',
 'Stroud',
 'Strouds',
 'Suddenly',
 'That',
 'The',
 'Then',
 'There',
 'They',
 'This',
 'Those',
 'Though',
 'Thwing',
 'Thwings',
 'To',
 'Usually',
 'Venetian',
 'Victor',
 '

In [222]:
vocab = {t: i for i, t in enumerate(all_tokens)}

In [223]:
vocab

{'!': 0,
 '"': 1,
 "'": 2,
 '(': 3,
 ')': 4,
 ',': 5,
 '--': 6,
 '.': 7,
 ':': 8,
 ';': 9,
 '?': 10,
 'A': 11,
 'Ah': 12,
 'Among': 13,
 'And': 14,
 'Are': 15,
 'Arrt': 16,
 'As': 17,
 'At': 18,
 'Be': 19,
 'Begin': 20,
 'Burlington': 21,
 'But': 22,
 'By': 23,
 'Carlo': 24,
 'Chicago': 25,
 'Claude': 26,
 'Come': 27,
 'Croft': 28,
 'Destroyed': 29,
 'Devonshire': 30,
 'Don': 31,
 'Dubarry': 32,
 'Emperors': 33,
 'Florence': 34,
 'For': 35,
 'Gallery': 36,
 'Gideon': 37,
 'Gisburn': 38,
 'Gisburns': 39,
 'Grafton': 40,
 'Greek': 41,
 'Grindle': 42,
 'Grindles': 43,
 'HAD': 44,
 'Had': 45,
 'Hang': 46,
 'Has': 47,
 'He': 48,
 'Her': 49,
 'Hermia': 50,
 'His': 51,
 'How': 52,
 'I': 53,
 'If': 54,
 'In': 55,
 'It': 56,
 'Jack': 57,
 'Jove': 58,
 'Just': 59,
 'Lord': 60,
 'Made': 61,
 'Miss': 62,
 'Money': 63,
 'Monte': 64,
 'Moon-dancers': 65,
 'Mr': 66,
 'Mrs': 67,
 'My': 68,
 'Never': 69,
 'No': 70,
 'Now': 71,
 'Nutley': 72,
 'Of': 73,
 'Oh': 74,
 'On': 75,
 'Once': 76,
 'Only': 77,
 '

In [224]:
vocab.items()

dict_items([('!', 0), ('"', 1), ("'", 2), ('(', 3), (')', 4), (',', 5), ('--', 6), ('.', 7), (':', 8), (';', 9), ('?', 10), ('A', 11), ('Ah', 12), ('Among', 13), ('And', 14), ('Are', 15), ('Arrt', 16), ('As', 17), ('At', 18), ('Be', 19), ('Begin', 20), ('Burlington', 21), ('But', 22), ('By', 23), ('Carlo', 24), ('Chicago', 25), ('Claude', 26), ('Come', 27), ('Croft', 28), ('Destroyed', 29), ('Devonshire', 30), ('Don', 31), ('Dubarry', 32), ('Emperors', 33), ('Florence', 34), ('For', 35), ('Gallery', 36), ('Gideon', 37), ('Gisburn', 38), ('Gisburns', 39), ('Grafton', 40), ('Greek', 41), ('Grindle', 42), ('Grindles', 43), ('HAD', 44), ('Had', 45), ('Hang', 46), ('Has', 47), ('He', 48), ('Her', 49), ('Hermia', 50), ('His', 51), ('How', 52), ('I', 53), ('If', 54), ('In', 55), ('It', 56), ('Jack', 57), ('Jove', 58), ('Just', 59), ('Lord', 60), ('Made', 61), ('Miss', 62), ('Money', 63), ('Monte', 64), ('Moon-dancers', 65), ('Mr', 66), ('Mrs', 67), ('My', 68), ('Never', 69), ('No', 70), ('Now

- Below are the first 50 entries in this vocabulary:

In [225]:
for item in vocab.items():
    print(item)
    if item[1] >= 50:
        break

# Expected output:
# ('!', 0)
# ('"', 1)
# ("'", 2)
# ('(', 3)
# (')', 4)
# (',', 5)
# ('--', 6)
# ('.', 7)
# (':', 8)
# (';', 9)
# ('?', 10)
# ('A', 11)
# ('Ah', 12)
# ('Among', 13)
# ('And', 14)
# ('Are', 15)
# ('Arrt', 16)
# ('As', 17)
# ('At', 18)
# ('Be', 19)
# ('Begin', 20)
# ('Burlington', 21)
# ('But', 22)
# ('By', 23)
# ('Carlo', 24)
# ('Chicago', 25)
# ('Claude', 26)
# ('Come', 27)
# ('Croft', 28)
# ('Destroyed', 29)
# ('Devonshire', 30)
# ('Don', 31)
# ('Dubarry', 32)
# ('Emperors', 33)
# ('Florence', 34)
# ('For', 35)
# ('Gallery', 36)
# ('Gideon', 37)
# ('Gisburn', 38)
# ('Gisburns', 39)
# ('Grafton', 40)
# ('Greek', 41)
# ('Grindle', 42)
# ('Grindles', 43)
# ('HAD', 44)
# ('Had', 45)
# ('Hang', 46)
# ('Has', 47)
# ('He', 48)
# ('Her', 49)
# ('Hermia', 50)

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- Below, we illustrate the tokenization of a short sample text using a small vocabulary:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

- Putting it now all together into a tokeniser class

In [226]:
class SimpletokeniserV1:
    def __init__(self, vocab):
        self.token_to_index = vocab
        self.index_to_token = {i:t for t,i in vocab.items()}

    def encode(self, text):
        tokens = [s for s in re.split(r'(\s|[,.:;?_!"()\']|--)', text) if s.strip()]
        return [self.token_to_index[t] for t in tokens]


    def decode(self, ids):
        words = [self.index_to_token[i] for i in ids]
        text = " ".join(words)
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

- The `encode` function turns text into token IDs
- The `decode` function turns token IDs back into text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

- We can use the tokeniser to encode (that is, tokenize) texts into integers
- These integers can then be embedded (later) as input of/for the LLM

In [227]:
tokeniser = SimpletokeniserV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""

ids = tokeniser.encode(text)

print(ids)

# Expected output:
# [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- We can decode the integers back into text

In [228]:
tokens = tokeniser.decode(ids)
print(tokens)

# Expected output:
# '" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [229]:
tokeniser.decode(tokeniser.encode(text))

# Expected output:
# '" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

&nbsp;
## 2.4 Adding special context tokens

- It's useful to add some "special" tokens for unknown words and to denote the end of a text

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- Some tokenisers use special tokens to help the LLM with additional context
- Some of these special tokens are
  - `[BOS]` (beginning of sequence) marks the beginning of text
  - `[EOS]` (end of sequence) marks where the text ends (this is usually used to concatenate multiple unrelated texts, e.g., two different Wikipedia articles or two different books, and so on)
  - `[PAD]` (padding) if we train LLMs with a batch size greater than 1 (we may include multiple texts with different lengths; with the padding token we pad the shorter texts to the longest length so that all texts have an equal length)
- `[UNK]` to represent words that are not included in the vocabulary

- Note that GPT-2 does not need any of these tokens mentioned above but only uses an `<|endoftext|>` token to reduce complexity
- The `<|endoftext|>` is analogous to the `[EOS]` token mentioned above
- GPT also uses the `<|endoftext|>` for padding (since we typically use a mask when training on batched inputs, we would not attend padded tokens anyways, so it does not matter what these tokens are)
- GPT-2 does not use an `<UNK>` token for out-of-vocabulary words; instead, GPT-2 uses a byte-pair encoding (BPE) tokeniser, which breaks down words into subword units which we will discuss in a later section



- We use the `<|endoftext|>` tokens between two independent sources of text:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

- Let's see what happens if we tokenize the following text:

In [230]:
text = "Hello, do you like tea. Is this-- a test?"

tokeniser.encode(text)


# Expected output:
# ---------------------------------------------------------------------------
# KeyError                                  Traceback (most recent call last)
# Cell In[17], line 5
#       1 tokeniser = SimpletokeniserV1(vocab)
#       3 text = "Hello, do you like tea. Is this-- a test?"
# ----> 5 tokeniser.encode(text)
#
# Cell In[13], line 12, in SimpletokeniserV1.encode(self, text)
#       7 preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
#       9 preprocessed = [
#      10     item.strip() for item in preprocessed if item.strip()
#      11 ]
# ---> 12 ids = [self.str_to_int[s] for s in preprocessed]
#      13 return ids
#
# Cell In[13], line 12, in <listcomp>(.0)
#       7 preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
#       9 preprocessed = [
#      10     item.strip() for item in preprocessed if item.strip()
#      11 ]
# ---> 12 ids = [self.str_to_int[s] for s in preprocessed]
#      13 return ids
#
# KeyError: 'Hello'

KeyError: 'Hello'

- The above produces an error because the word "Hello" is not contained in the vocabulary
- To deal with such cases, we can add special tokens like `"<|unk|>"` to the vocabulary to represent unknown words
- Since we are already extending the vocabulary, let's add another token called `"<|endoftext|>"` which is used in GPT-2 training to denote the end of a text (and it's also used between concatenated text, like if our training datasets consists of multiple articles, books, etc.)

In [231]:
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: idx for idx, token in enumerate(all_tokens)}

In [232]:
len(vocab)
# Expected output:
# 1132

1132

In [233]:
list(vocab.items())[-5:]
# Expected output:
# ('younger', 1127)
# ('your', 1128)
# ('yourself', 1129)
# ('<|endoftext|>', 1130)
# ('<|unk|>', 1131)

[('younger', 1127),
 ('your', 1128),
 ('yourself', 1129),
 ('<|endoftext|>', 1130),
 ('<|unk|>', 1131)]

- We also need to adjust the tokeniser accordingly so that it knows when and how to use the new `<unk>` token

In [236]:
class SimpletokeniserV2:
    def __init__(self, vocab):
        self.token_to_index = vocab
        self.index_to_token = {i:t for t,i in vocab.items()}

    def encode(self, text):
        preprocessed = [s for s in re.split(r'(\s|[,.:;?_!"()\']|--)', text) if s.strip()]
        preprocessed = [s if s in self.token_to_index else '<|unk|>' for s in preprocessed]
        return [self.token_to_index[t] for t in preprocessed]


    def decode(self, ids):
        words = [self.index_to_token[i] for i in ids]
        text = " ".join(words)
        return re.sub(r'\s+([,.?!"()\'])', r'\1', text)

Let's try to tokenize text with the modified tokenizer:

In [237]:
tokeniser_v2 = SimpletokeniserV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

# Expected output:
# Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [238]:
tokeniser_v2.encode(text)

# Expected output:
# [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [239]:
tokeniser_v2.decode(tokeniser_v2.encode(text))
# Expected output:
# '<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

&nbsp;
## 2.5 BytePair encoding

- GPT-2 used BytePair encoding (BPE) as its tokenizer
- it allows the model to break down words that aren't in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words
- For instance, if GPT-2's vocabulary doesn't have the word "unfamiliarword," it might tokenize it as ["unfam", "iliar", "word"] or some other subword breakdown, depending on its trained BPE merges
- The original BPE tokenizer can be found here: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- In this chapter, we are using the BPE tokenizer from OpenAI's open-source [tiktoken](https://github.com/openai/tiktoken) library, which implements its core algorithms in Rust to improve computational performance
- I created a notebook in the [./bytepair_encoder](../02_bonus_bytepair-encoder) that compares these two implementations side-by-side (tiktoken was about 5x faster on the sample text)

In [ ]:
# pip install tiktoken

In [278]:
import importlib
import tiktoken

print("tiktoken version: ", version("tiktoken"))
# Expected output:
# tiktoken version: 0.7.0

tiktoken version:  0.13.0


In [242]:
tokeniser = tiktoken.get_encoding("gpt2")

In [245]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokeniser.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

# Expected output:
# [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [246]:
tokeniser.decode(integers)

# Expected output:
# Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.

'Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.'

- BPE tokenizers break down unknown words into subwords and individual characters:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

## 2.6 Data sampling with a sliding window

- We train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [247]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokeniser.encode(raw_text)
print(len(enc_text))

# Expected output:
# 5145

5145


- For each text chunk, we want the inputs and targets
- Since we want the model to predict the next word, the targets are the inputs shifted by one position to the right

In [249]:
enc_sample = enc_text[50:]
print(enc_sample)

[290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 34686, 41976, 13, 357, 10915, 314, 2138, 1807, 340, 561, 423, 587, 10598, 393, 28537, 2014, 198, 198, 1, 464, 6001, 286, 465, 13476, 1, 438, 5562, 373, 644, 262, 1466, 1444, 340, 13, 314, 460, 3285, 9074, 13, 46606, 536, 5469, 438, 14363, 938, 4842, 1650, 353, 438, 2934, 489, 3255, 465, 48422, 540, 450, 67, 3299, 13, 366, 5189, 1781, 340, 338, 1016, 284, 3758, 262, 1988, 286, 616, 4286, 705, 1014, 510, 26, 475, 314, 836, 470, 892, 286, 326, 11, 1770, 13, 8759, 2763, 438, 1169, 2994, 284, 943, 17034, 318, 477, 314, 892, 286, 526, 383, 1573, 11, 319, 9074, 13, 536, 5469, 338, 11914, 11, 33096, 663, 4808, 3808, 62, 355, 996, 484, 547, 12548, 287, 281, 13079, 410, 12523, 286, 22353, 13, 843, 340, 373, 407, 691, 262, 9074, 13, 536, 48819, 508, 25722, 276, 13, 11161, 407, 262, 40123, 18113, 544, 9325, 701, 11, 379, 262, 938, 402, 1617, 261, 12917, 905, 11, 5025, 502, 878, 402, 271, 10899, 338, 366, 31640, 12, 67, 20811, 1, 284, 910, 11, 351, 10

In [270]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(x)
print(y)

# Expected output:
# x: [290, 4920, 2241, 287]
# y:      [4920, 2241, 287, 257]

[290, 4920, 2241, 287]
[4920, 2241, 287, 257]


- One by one, the prediction would look like as follows:

In [272]:
for i in range(context_size):
    input = x[:i+1]
    output = y[i]
    print(input, "-->", output)

 
# Expected output:
# [290] ----> 4920
# [290, 4920] ----> 2241
# [290, 4920, 2241] ----> 287
# [290, 4920, 2241, 287] ----> 257

[290] --> 4920
[290, 4920] --> 2241
[290, 4920, 2241] --> 287
[290, 4920, 2241, 287] --> 257


In [274]:
for i in range(context_size):
    input = x[:i+1]
    output = y[i]
    # print(tokeniser.decode(list(input)))
    # print(tokeniser.decode([output]))
    print(tokeniser.decode(list(input)), "-->", tokeniser.decode([output]))
    

# Expected output:
#  and ---->  established
#  and established ---->  himself
#  and established himself ---->  in
#  and established himself in ---->  a

 and -->  established
 and established -->  himself
 and established himself -->  in
 and established himself in -->  a


- We will take care of the next-word prediction in a later chapter after we covered the attention mechanism
- For now, we implement a simple data loader that iterates over the input dataset and returns the inputs and targets shifted by one

- Install and import PyTorch (see Appendix A for installation tips)

In [277]:
import torch

print(version("torch"))

# Expected output:
# PyTorch version: 2.5.1

2.5.1


- We use a sliding window approach, changing the position by +1:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

- Create dataset and dataloader that extract chunks from the input text dataset

In [321]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokeniser, max_length, stride=1):
        self.input_ids = []
        self.target_ids = []
        
        # Tokenize the entire text
        ids = tokeniser.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        # x: [290, 4920, 2241, 287]
        # y:      [4920, 2241, 287, 257]
        for i in range(0, len(ids) - max_length, stride):
            x = ids[i:i + max_length]
            y = ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(x))
            self.target_ids.append(torch.tensor(y))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [322]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokeniser = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokeniser, max_length, stride)

    # Create dataloader
    return DataLoader(dataset, batch_size, shuffle, num_workers=num_workers, drop_last=drop_last)

- Let's test the dataloader with a batch size of 1 for an LLM with a context size of 4:

In [312]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

In [313]:
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

# Expected output:
# [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [314]:
first_batch[0].shape

torch.Size([1, 4])

In [315]:
second_batch = next(data_iter)
print(second_batch)

# Expected output:
# [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- An example using stride equal to the context length (here: 4) as shown below:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- We can also create batched outputs
- Note that we increase the stride here so that we don't have overlaps between the batches, since more overlap could lead to increased overfitting

In [323]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)


# Expected output:
# Inputs:
#  tensor([[   40,   367,  2885,  1464],
#         [ 1807,  3619,   402,   271],
#         [10899,  2138,   257,  7026],
#         [15632,   438,  2016,   257],
#         [  922,  5891,  1576,   438],
#         [  568,   340,   373,   645],
#         [ 1049,  5975,   284,   502],
#         [  284,  3285,   326,    11]])
#
# Targets:
#  tensor([[  367,  2885,  1464,  1807],
#         [ 3619,   402,   271, 10899],
#         [ 2138,   257,  7026, 15632],
#         [  438,  2016,   257,   922],
#         [ 5891,  1576,   438,   568],
#         [  340,   373,   645,  1049],
#         [ 5975,   284,   502,   284],
#         [ 3285,   326,    11,   287]])

[tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]]), tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])]


&nbsp;
## 2.7 Creating token embeddings

- The data is already almost ready for an LLM
- But lastly let us embed the tokens in a continuous vector representation using an embedding layer
- Usually, these embedding layers are part of the LLM itself and are updated (trained) during model training

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

- Suppose we have the following four input examples with input ids 2, 3, 5, and 1 (after tokenization):

In [324]:
input_ids = torch.tensor([2, 3, 5, 1])

- For the sake of simplicity, suppose we have a small vocabulary of only 6 words and we want to create embeddings of size 3:

In [325]:
vocab_size = 6
emb_size = 3
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, emb_size)

- This would result in a 6x3 weight matrix:

In [ ]:
print(embedding_layer.weight)

# Expected output:
# Parameter containing:
# tensor([[ 0.3374, -0.1778, -0.1690],
#         [ 0.9178,  1.5810,  1.3010],
#         [ 1.2753, -0.2010, -0.1606],
#         [-0.4015,  0.9666, -1.1481],
#         [-1.1589,  0.3255, -0.6315],
#         [-2.8400, -0.7849, -1.4096]], requires_grad=True)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


- For those who are familiar with one-hot encoding, the embedding layer approach above is essentially just a more efficient way of implementing one-hot encoding followed by matrix multiplication in a fully-connected layer, which is described in the supplementary code in [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul)
- Because the embedding layer is just a more efficient implementation that is equivalent to the one-hot encoding and matrix-multiplication approach it can be seen as a neural network layer that can be optimized via backpropagation

- To convert a token with id 3 into a 3-dimensional vector, we do the following:

In [332]:
embedding_layer.weight[3]

tensor([-0.4015,  0.9666, -1.1481], grad_fn=<SelectBackward0>)

In [330]:
embedding_layer(torch.tensor([3]))

# Expected output:
# tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)

- Note that the above is the 4th row in the `embedding_layer` weight matrix
- To embed all four `input_ids` values above, we do

In [ ]:
embedding_layer(input_ids)

# Expected output:
# tensor([[ 1.2753, -0.2010, -0.1606],
#         [-0.4015,  0.9666, -1.1481],
#         [-2.8400, -0.7849, -1.4096],
#         [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)

- An embedding layer is essentially a look-up operation:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

- **You may be interested in the bonus content comparing embedding layers with regular linear layers: [../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**

&nbsp;
## 2.8 Encoding word positions

- Embedding layer convert IDs into identical vector representations regardless of where they are located in the input sequence:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

- Positional embeddings are combined with the token embedding vector to form the input embeddings for a large language model:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- The BytePair encoder has a vocabulary size of 50,257:
- Suppose we want to encode the input tokens into a 256-dimensional vector representation:

In [335]:
vocab_size = 50257
output_dim = 256

embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- If we sample data from the dataloader, we embed the tokens in each batch into a 256-dimensional vector
- If we have a batch size of 8 with 4 tokens each, this results in a 8 x 4 x 256 tensor:

In [341]:
max_length=4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [343]:
print(inputs)
print(inputs.shape)
# Expected output:
# Token IDs:
#  tensor([[   40,   367,  2885,  1464],
#         [ 1807,  3619,   402,   271],
#         [10899,  2138,   257,  7026],
#         [15632,   438,  2016,   257],
#         [  922,  5891,  1576,   438],
#         [  568,   340,   373,   645],
#         [ 1049,  5975,   284,   502],
#         [  284,  3285,   326,    11]])
#
# Inputs shape:
#  torch.Size([8, 4])

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
torch.Size([8, 4])


In [347]:
token_embeddings = embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(token_embeddings)

# Expected output:
# torch.Size([8, 4, 256])

torch.Size([8, 4, 256])
tensor([[[ 0.4913,  1.1239,  1.4588,  ..., -0.3995, -1.8735, -0.1445],
         [ 0.4481,  0.2536, -0.2655,  ...,  0.4997, -1.1991, -1.1844],
         [-0.2507, -0.0546,  0.6687,  ...,  0.9618,  2.3737, -0.0528],
         [ 0.9457,  0.8657,  1.6191,  ..., -0.4544, -0.7460,  0.3483]],

        [[ 1.5460,  1.7368, -0.7848,  ..., -0.1004,  0.8584, -0.3421],
         [-1.8622, -0.1914, -0.3812,  ...,  1.1220, -0.3496,  0.6091],
         [ 1.9847, -0.6483, -0.1415,  ..., -0.3841, -0.9355,  1.4478],
         [ 0.9647,  1.2974, -1.6207,  ...,  1.1463,  1.5797,  0.3969]],

        [[-0.7713,  0.6572,  0.1663,  ..., -0.8044,  0.0542,  0.7426],
         [ 0.8046,  0.5047,  1.2922,  ...,  1.4648,  0.4097,  0.3205],
         [ 0.0795, -1.7636,  0.5750,  ...,  2.1823,  1.8231, -0.3635],
         [ 0.4267, -0.0647,  0.5686,  ..., -0.5209,  1.3065,  0.8473]],

        ...,

        [[-1.6156,  0.9610, -2.6437,  ..., -0.9645,  1.0888,  1.6383],
         [-0.3985, -0.9235, -1.31

- GPT-2 uses absolute position embeddings, so we just create another embedding layer:

In [350]:
pos_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
print(pos_embedding_layer.weight)

Parameter containing:
tensor([[-0.7718,  1.2198, -0.4126,  ...,  1.4056,  0.2723,  1.2280],
        [-0.0625,  0.3353, -0.6602,  ..., -0.9091, -1.2925, -0.4175],
        [-0.3727,  0.1802, -0.5434,  ..., -1.0483, -0.3196,  0.8705],
        ...,
        [ 0.1873,  0.2767, -0.2924,  ...,  0.0649, -0.4011, -0.0205],
        [-1.7190,  0.4360, -1.3934,  ..., -1.3885, -0.4998,  1.0916],
        [-0.7563, -0.0071, -0.2751,  ...,  0.1130, -0.2909, -0.7629]],
       requires_grad=True)


In [354]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(pos_embeddings)

# Expected output:
# torch.Size([4, 256])

torch.Size([4, 256])
tensor([[-0.7718,  1.2198, -0.4126,  ...,  1.4056,  0.2723,  1.2280],
        [-0.0625,  0.3353, -0.6602,  ..., -0.9091, -1.2925, -0.4175],
        [-0.3727,  0.1802, -0.5434,  ..., -1.0483, -0.3196,  0.8705],
        [ 1.4247,  0.0735,  0.6832,  ..., -0.0169,  0.9855, -0.9120]],
       grad_fn=<EmbeddingBackward0>)


- To create the input embeddings used in an LLM, we simply add the token and the positional embeddings:

In [356]:
input_embeddings = pos_embeddings + token_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
print(input_embeddings)

# Expected output:
# torch.Size([8, 4, 256])

torch.Size([8, 4, 256])
tensor([[[-0.2805,  2.3437,  1.0463,  ...,  1.0060, -1.6013,  1.0834],
         [ 0.3856,  0.5890, -0.9257,  ..., -0.4094, -2.4916, -1.6019],
         [-0.6233,  0.1255,  0.1254,  ..., -0.0866,  2.0541,  0.8177],
         [ 2.3704,  0.9392,  2.3023,  ..., -0.4714,  0.2395, -0.5636]],

        [[ 0.7742,  2.9567, -1.1973,  ...,  1.3052,  1.1307,  0.8859],
         [-1.9247,  0.1440, -1.0414,  ...,  0.2129, -1.6421,  0.1916],
         [ 1.6121, -0.4681, -0.6848,  ..., -1.4324, -1.2551,  2.3183],
         [ 2.3894,  1.3709, -0.9376,  ...,  1.1293,  2.5652, -0.5151]],

        [[-1.5432,  1.8770, -0.2463,  ...,  0.6012,  0.3264,  1.9705],
         [ 0.7421,  0.8400,  0.6320,  ...,  0.5557, -0.8829, -0.0969],
         [-0.2932, -1.5834,  0.0316,  ...,  1.1339,  1.5035,  0.5070],
         [ 1.8514,  0.0088,  1.2517,  ..., -0.5378,  2.2920, -0.0647]],

        ...,

        [[-2.3874,  2.1808, -3.0563,  ...,  0.4411,  1.3611,  2.8663],
         [-0.4610, -0.5882, -1.97

- In the initial phase of the input processing workflow, the input text is segmented into separate tokens
- Following this segmentation, these tokens are transformed into token IDs based on a predefined vocabulary:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">

&nbsp;
## Summary and takeaways

See the [./dataloader.ipynb](./dataloader.ipynb) code notebook, which is a concise version of the data loader that we implemented in this chapter and will need for training the GPT model in upcoming chapters.

See [./exercise-solutions.ipynb](./exercise-solutions.ipynb) for the exercise solutions.

See the [Byte Pair Encoding (BPE) Tokenizer From Scratch](../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb) notebook if you are interested in learning how the GPT-2 tokenizer can be implemented and trained from scratch.